# Activation Functions
Activation functions introduce non-linearity, allowing neural networks to learn complex decision boundaries.

## Why They Matter
- Shape the final output distribution 
- Help decide whether a neuron should activate for a given input.
- Influence gradient flow during backpropagation, impacting training stability.

## Common Activation Families
1. **Step**: Outputs 0 or 1; historically used in perceptrons but not differentiable.
2. **Sigmoid**: Maps inputs to (0, 1); useful for binary probabilities but can saturate.
3. **Tanh**: Zero-centered version of sigmoid with outputs in (-1, 1).
4. **ReLU**: Fast and sparse; outputs max(0, x) and scales to large magnitudes.
5. **Leaky ReLU**: Adds a small slope for negative inputs to reduce dead-neuron issues.

In [9]:
def stepfunction(x):
    return 1 if x >= 0 else 0

def sigmoid(x):
    import math
    return 1 / (1 + math.exp(-x))

def tanh(x):
    import math
    return math.tanh(x)

def relu(x):
    return max(0, x)

def leaky_relu(x, alpha=0.01):
    return x if x >= 0 else alpha * x



# Loss vs. Cost Functions
- **Loss function**: Measures the error for a single example.
- **Cost function**: Aggregates loss across the dataset (mean or sum) to guide optimization.

Understanding these metrics is essential for adjusting weights and biases—lower cost implies better alignment between predictions and targets.

In [ ]:
# Classification tasks favor MAE when robustness to outliers matters.
def mean_absolute_error(y_true, y_pred):
    return sum(abs(t - p) for t, p in zip(y_true, y_pred)) / len(y_true)

# Regression baselines often start with MSE to heavily penalize large errors.
def mean_squared_error(y_true, y_pred):
    return sum((t - p) ** 2 for t, p in zip(y_true, y_pred)) / len(y_true)

# RMSE restores the original scale, making interpretation easier for stakeholders.
def root_mean_squared_error(y_true, y_pred):
    import math
    return math.sqrt(mean_squared_error(y_true, y_pred))

# Binary cross-entropy (log loss) is the go-to objective for probabilistic classifiers.
def log_loss(y_true, y_pred):
    import math
    epsilon = 1e-15
    y_pred = [min(max(p, epsilon), 1 - epsilon) for p in y_pred]
    return -sum(t * math.log(p) + (1 - t) * math.log(1 - p) for t, p in zip(y_true, y_pred)) / len(y_true)

# Sparse categorical cross-entropy handles integer labels for multi-class problems.
def sparse_categorical_crossentropy(y_true, y_pred):
    import math
    epsilon = 1e-15
    y_pred = [min(max(p, epsilon), 1 - epsilon) for p in y_pred]
    return -sum(math.log(y_pred[t]) for t, p in zip(y_true, y_pred)) / len(y_true)



# Gradient Descent (Heart of Neural Networks)
Gradient descent iteratively nudges weights and biases toward the minimum of the cost surface

Update rule for a weight $w$:
- $w = w - \alpha \cdot \frac{\partial \mathcal{L}}{\partial w}$
- $\alpha$ is the learning rate that controls step size.

The goal is to converge on a global or useful local minimum improving accuracy over time

In [ ]:
import numpy as np
def batch_gradient_descent( X, y, learning_rate=0.01, epochs=1000):
        m, n = X.shape
        weights = np.zeros(n)
        bias = 0

        for epoch in range(epochs):
            weighted_sum = np.dot(X, weights) + bias
            y_pred = self.sigmoid(weighted_sum)
            log_loss_value = self._log_loss(y, y_pred)
            error = y_pred - y
            dw = (1/m) * np.dot(np.transpose(X), error)
            db = (1/m) * np.sum(error)
            weights -= learning_rate * dw
            bias -= learning_rate * db

            print(f'Epoch {epoch+1}/{epochs}, Log Loss: {log_loss_value}')

        return weights, bias

# Single-Layer Neural Network From Scratch


In [27]:
import numpy as np
import pandas as pd

class NeuralNetwork:
    def __init__(self):
        self.weights = None
        self.bias = 0

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def stepfunction(self, y_pred):
        return np.where(y_pred >= 0.5, 1, 0)

    def _log_loss(self, y_true, y_pred, eps=1e-15):
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    def accuracy(self, y_true, y_pred):
        y_pred_labels = self.stepfunction(y_pred)
        return np.mean(y_true == y_pred_labels)

    def batch_gradient_descent(self, X, y, learning_rate=0.01, epochs=1000):
        X = np.array(X)
        y = np.array(y)

        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0

        for epoch in range(epochs):
            weighted_sum = np.dot(X, self.weights) + self.bias
            y_pred = self.sigmoid(weighted_sum)

            loss = self._log_loss(y, y_pred)

            error = y_pred - y
            dw = (1/m) * np.dot(X.T, error)
            db = (1/m) * np.sum(error)

            self.weights -= learning_rate * dw
            self.bias -= learning_rate * db

            acc = self.accuracy(y, y_pred)
            print(f"Epoch {epoch+1}/{epochs}, Log Loss: {loss:.4f}, Accuracy: {acc:.4f}")

        return self.weights, self.bias

    def fit(self, X, y, method='batch', learning_rate=0.01, epochs=1000, loss_function='log_loss'):
        if method == 'batch':
            return self.batch_gradient_descent(X, y, learning_rate, epochs)

    def predict(self, X):
        X = np.array(X)
        y_pred = self.sigmoid(np.dot(X, self.weights) + self.bias)
        return self.stepfunction(y_pred)

